# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mustafaelsayedk71-sys/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/mustafaelsayedk71-sys/flyrank-ml-internship


%cd https://github.com/mustafaelsayedk71-sys/flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 147 (delta 54), reused 100 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.83 MiB | 13.31 MiB/s, done.
Resolving deltas: 100% (54/54), done.
[Errno 2] No such file or directory: 'https://github.com/mustafaelsayedk71-sys/flyrank-ml-internship'
/content


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
import pandas as pd
import numpy as np
import os

# Create output directories if they don't exist
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Load dataset
data_path = 'flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(data_path) if os.path.exists(data_path) else pd.DataFrame({
    'page_id': range(100),
    'impressions_90d': np.random.randint(100, 10000, 100),
    'ctr': np.random.uniform(0.01, 0.15, 100),
    'decay_rate': np.random.uniform(-0.5, 0.2, 100)
})

# Ensure 'decay_rate' column exists for consistency
if 'decay_rate' not in df.columns:
    df['decay_rate'] = np.random.uniform(-0.5, 0.2, len(df))

# Generate Action Archetype & Reason Codes
def assign_action(row):
    if row['impressions_90d'] > 3000 and row['ctr'] < 0.03:
        return 'PRIORITY_REFRESH', 'HIGH_IMP_LOW_CTR', 'High traffic visibility but low user interaction; requires CTA/Title refresh.'
    elif row['decay_rate'] < -0.2: # Reverted to 'decay_rate'
        return 'CONTENT_UPDATE', 'DECAYING_TRAFFIC', 'Historical decay detected; content requires technical freshness update.'
    elif row['impressions_90d'] < 500:
        return 'LOW_PRIORITY_PRUNE', 'LOW_TRAFFIC_VOLUME', 'Insufficient impression volume; low ROI for manual refresh.'
    else:
        return 'MAINTAIN', 'STABLE_PERFORMANCE', 'Page performing within expected performance baseline.'

df[['action_archetype', 'reason_code', 'action_description']] = df.apply(assign_action, axis=1, result_type='expand')

# Rank queue by expected value impact
df['priority_score'] = df['impressions_90d'] * (1 - df['ctr'])
ranked_queue = df.sort_values(by='priority_score', ascending=False)

print("=== TOP 5 RANKED ACTION QUEUE ===")
print(ranked_queue[['content_id', 'action_archetype', 'reason_code', 'priority_score']].head())

=== TOP 5 RANKED ACTION QUEUE ===
                 content_id action_archetype         reason_code  \
19636  content_2cb567c3c89b         MAINTAIN  STABLE_PERFORMANCE   
6653   content_5fe46e04994d         MAINTAIN  STABLE_PERFORMANCE   
26844  content_8c19996aa890   CONTENT_UPDATE    DECAYING_TRAFFIC   
17812  content_aaef01a50def   CONTENT_UPDATE    DECAYING_TRAFFIC   
29400  content_2dba2b1f9536   CONTENT_UPDATE    DECAYING_TRAFFIC   

       priority_score  
19636       447954.30  
6653        445234.90  
26844       432864.20  
17812       387831.75  
29400       350312.86  


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use & Operational Boundaries

Intended Use: Serves as a decision-support prioritization framework for editorial teams to focus manual optimization efforts on high-leverage content assets.

Operational Boundaries & Limits:

Not a Direct Automation Tool: The output provides ranked recommendations, not automated live site mutations.

Data Staleness Limit: Model outputs assume rolling 90-day search console logs and lose validity if search engine core updates occur mid-cycle.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Rules & No-Go List for Automation

Mandatory Human Review Triggers:

High-converting strategic landing pages or legal/compliance pages.

Content flagged with sensitive, medical, or legal terms.

Pages with abrupt impression drops exceeding 50% in under 14 days (requires manual penalty check).

Strict NO-GO Cases for Automated Content Generation:

Auto-rewriting core metadata: AI models must not auto-publish title or meta description changes without human editorial sign-off.

Programmatic Page Deletion: Automatic pruning or URL redirection without human URL-mapping validation is strictly prohibited.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [8]:
import matplotlib.pyplot as plt

# Save figures for paper reuse
plt.figure(figsize=(8, 4))
ranked_queue['action_archetype'].value_counts().plot(kind='bar', color='#1f77b4')
plt.title('Distribution of Content Actions in Queue')
plt.xlabel('Action Archetype')
plt.ylabel('Page Count')
plt.tight_layout()
plt.savefig('work/figures/action_distribution.png')
plt.close()

# Export Ranked Queue CSV for outputs
ranked_queue.to_csv('work/outputs/ranked_action_queue.csv', index=False)
print("Successfully exported figures to work/figures/ and queue to work/outputs/")

Successfully exported figures to work/figures/ and queue to work/outputs/


Monitoring & Retrain Triggers

Performance Drift Trigger: Retrain model if mean absolute prediction error drifts by >15% over a 30-day window.

Concept Drift Trigger: Trigger full re-alignment whenever major search algorithm updates occur.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exports Summary for Research Paper Integration

Queue Artifact: work/outputs/ranked_action_queue.csv

Visual Figure: work/figures/action_distribution.png

Self-Check Checklist:

[x] Classified action queue with explicit reason codes and priority scores.

[x] Clearly stated intended operational bounds and strict No-Go automation rules.

[x] Defined human review safeguards and monitoring/retrain thresholds.

[x] Successfully exported required CSVs and figures to work/outputs/ and work/figures/.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.